# Dataset Inspection & Cleaning Template

A fill-in-the-blanks checklist for checking whether the dataset **you** picked is actually usable for a machine learning project.

This follows the workflow from *Machine Learning Part II* (Section 03, "Data Processing & Visualization"): load it, inspect it in the first ten minutes, visualize it, hunt for the six classic problems, then get an honest go / no-go verdict before you spend a single hour modelling.

**How to use this notebook:**
1. Copy this file and rename it for your project.
2. Fill in the **Config** cell in Section 0 with your own file path/URL and target column.
3. Run every cell top to bottom (`Restart & Run All` before you trust any result — see Part II, slide 21).
4. Read the printed messages under each check — they tell you what to do next, not just what's wrong.
5. The last section prints a scored verdict. If it says **not ready**, fix what it flags and re-run before moving on to modelling.

## 0. Setup and Config

Run the imports cell once, then fill in the config cell below with your own dataset's details. Nothing else in the notebook needs to change — every check further down reads from this config.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (7, 4.5)

print("Environment ready.")

In [ ]:
# =========================== EDIT THIS CELL ===========================

# Where is your file? Any of these work:
#   "data/my_dataset.csv"                    (local CSV)
#   "https://example.com/data.csv"           (a URL works anywhere a path does)
#   "data/my_dataset.xlsx"                   (Excel -- needs `pip install openpyxl`)
#   "data/my_dataset.json"                   (JSON)
#
# A ready-to-try example ships with this notebook:
#   "data/sample_student_dataset.csv"  (target column: "churn", problem type: "classification")
FILE_PATH = "data/my_dataset.csv"

# What format is it? one of: "csv", "excel", "json", "sql"
FILE_FORMAT = "csv"

# The column you eventually want to predict (your `y`). Leave as None if you
# don't have a target yet / this is for unsupervised learning.
TARGET_COLUMN = "target"

# Roughly what kind of problem is this? one of: "classification", "regression", "unsupervised"
# (Only used to pick which checks make sense later on -- see Part I, slide 12-14.)
PROBLEM_TYPE = "classification"

# Optional extra pandas read arguments, e.g. {"sep": ";", "encoding": "latin-1"}
# See Part II, slide 23 ("Common loading failures") if reading the file goes wrong.
READ_KWARGS = {}

# ========================================================================

print(f"Config set: {FILE_PATH!r} ({FILE_FORMAT}), target={TARGET_COLUMN!r}, problem={PROBLEM_TYPE!r}")

## 1. Loading the Dataset

`pandas` has one reader per format, and they all follow the same pattern: `pd.read_csv`, `pd.read_excel`, `pd.read_json`, `pd.read_sql`. The cell below picks the right one based on `FILE_FORMAT` and reports common failures in plain language instead of a raw traceback:

- `UnicodeDecodeError` → try adding `"encoding": "latin-1"` to `READ_KWARGS`.
- Everything landed in one column → wrong separator; try `"sep": ";"` (or `"\t"` for TSV).
- Numbers read as text → hidden symbols (currency signs, thousands commas) in the column.
- Dates read as strings → add `"parse_dates": ["your_date_column"]` to `READ_KWARGS`.
- `IsADirectoryError` → your `FILE_PATH` points at a **folder**, not a file. This is the normal return value of `kagglehub.dataset_download(...)` — it downloads the whole dataset into a folder and gives you that folder's path, not a specific CSV. The cell below will list what's actually inside the folder so you can pick the right file, e.g.:
  ```python
  import kagglehub
  path = kagglehub.dataset_download("mohamedhanyyy/video-games")
  # path is a FOLDER -- look inside it, then point FILE_PATH at the actual file:
  # FILE_PATH = path + "/vgsales.csv"
  ```

In [ ]:
import os


def load_dataset(path: str, fmt: str, **read_kwargs) -> pd.DataFrame:
    readers = {
        "csv": pd.read_csv,
        "excel": pd.read_excel,
        "json": pd.read_json,
    }
    if fmt not in readers:
        raise ValueError(f"Unsupported FILE_FORMAT {fmt!r}. Use one of: {list(readers)}")
    return readers[fmt](path, **read_kwargs)


try:
    df = load_dataset(FILE_PATH, FILE_FORMAT, **READ_KWARGS)
    print(f"Loaded successfully: {df.shape[0]:,} rows x {df.shape[1]} columns")
except IsADirectoryError:
    print(f"{FILE_PATH!r} is a FOLDER, not a file -- pandas needs the exact file inside it.")
    print("This happens a lot with kagglehub.dataset_download(), which returns a folder path,")
    print("not the CSV path. Files found inside that folder:\n")
    for entry in sorted(os.listdir(FILE_PATH)):
        print(f"  {os.path.join(FILE_PATH, entry)}")
    print("\nCopy the exact path of the file you want and set FILE_PATH to that, e.g.:")
    print(f'  FILE_PATH = "{FILE_PATH.rstrip(chr(47))}/<filename>.csv"')
    raise
except UnicodeDecodeError:
    print("UnicodeDecodeError -- try adding {'encoding': 'latin-1'} to READ_KWARGS and re-run.")
    raise
except FileNotFoundError:
    print(f"File not found at {FILE_PATH!r} -- check the path in the config cell above.")
    raise
except Exception as e:
    print(f"Loading failed: {type(e).__name__}: {e}")
    print("See the loading-failures list above for the most common causes.")
    raise

df.head(10)

## 2. Inspecting a Dataset — The First Ten Minutes

Never model data you have not looked at. Run this exact sequence on every dataset, every time — the habit matters more than the syntax.

| Command | Question it answers |
|---|---|
| `df.shape` | How many rows and columns am I working with? |
| `df.head(10)` | What does a record actually look like? |
| `df.info()` | Column names, dtypes, and non-null counts in one view. |
| `df.describe()` | Count, mean, std, min, quartiles, max for numeric columns. |
| `df.isna().sum()` | Missing values per column — the number that decides your cleaning plan. |
| `df.duplicated().sum()` | Repeated records that would bias every statistic. |
| `df[col].value_counts()` | Category frequencies; reveals class imbalance and typos. |
| `df.dtypes` | Are numbers stored as numbers, and dates as dates? |

**Then ask the question that matters:** does what you're seeing match what you were told this data contains? Mismatches here save weeks later.

In [ ]:
print("1. df.shape")
print(df.shape)

In [ ]:
print("2. df.head(10)")
df.head(10)

In [ ]:
print("3. df.info()")
df.info()

In [ ]:
print("4. df.describe() -- numeric columns only")
df.describe()

In [ ]:
print("5. df.isna().sum() -- missing values per column")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).sort_values(
    "missing_count", ascending=False
)

In [ ]:
print("6. df.duplicated().sum() -- repeated rows")
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {n_duplicates}")
if n_duplicates > 0:
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10))

In [ ]:
print("7. value_counts() for every non-numeric column -- checks class imbalance and typos")
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

if not categorical_cols:
    print("No categorical columns detected.")
else:
    for col in categorical_cols:
        n_unique = df[col].nunique(dropna=True)
        print(f"\n--- {col} ({n_unique} unique values) ---")
        if n_unique > 20:
            print(f"(more than 20 unique values -- showing top 10; possibly a free-text or ID column)")
            print(df[col].value_counts().head(10))
        else:
            print(df[col].value_counts())

In [ ]:
print("8. df.dtypes -- are numbers stored as numbers, and dates as dates?")
print(df.dtypes)

print("\nNow ask yourself: does this match what you were told this data contains?")
print("Look especially for: numeric-looking columns typed as 'object' (hidden symbols),")
print("and date-looking columns typed as 'object' instead of 'datetime64'.")

## 3. Visualizing to Understand

Summary statistics hide as much as they reveal — a chart shows shape, spread, and surprises in one look (see Anscombe's quartet if you want the classic proof: four datasets, identical means/variances/correlations, completely different shapes when plotted).

| Chart | Question it answers |
|---|---|
| Histogram | How is one numeric column distributed? Is it skewed? |
| Box plot | Where are the outliers, and how wide is the spread? |
| Scatter plot | Do these two variables move together? |
| Bar chart | How frequent is each category? |
| Heatmap | Which features correlate — and which are redundant? |

The cells below generate a histogram grid for every numeric column, a box plot for outlier-hunting, and a correlation heatmap — automatically, no editing needed even if your column names are different.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print(f"Numeric columns detected: {numeric_cols}")

# Histogram grid -- one panel per numeric column, checking shape and skew
if numeric_cols:
    n_cols = min(3, len(numeric_cols))
    n_rows = int(np.ceil(len(numeric_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.5 * n_rows))
    axes = np.atleast_1d(axes).flatten()
    for ax, col in zip(axes, numeric_cols):
        df[col].hist(bins=30, ax=ax)
        ax.set_title(col)
    for ax in axes[len(numeric_cols):]:
        ax.axis("off")
    fig.suptitle("Distribution of every numeric column")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns to plot.")

In [ ]:
# Box plots -- where are the outliers, and how wide is the spread?
if numeric_cols:
    fig, ax = plt.subplots(figsize=(max(6, len(numeric_cols) * 1.2), 4.5))
    df[numeric_cols].boxplot(ax=ax, rot=45)
    ax.set_title("Box plots -- look for points far outside the whiskers")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns to plot.")

In [ ]:
# Correlation heatmap -- which features move together, and which are redundant?
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(0.7 * len(numeric_cols) + 2, 0.7 * len(numeric_cols) + 2))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(numeric_cols)))
    ax.set_yticks(range(len(numeric_cols)))
    ax.set_xticklabels(numeric_cols, rotation=45, ha="right")
    ax.set_yticklabels(numeric_cols)
    for i in range(len(numeric_cols)):
        for j in range(len(numeric_cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, label="correlation")
    ax.set_title("Correlation heatmap")
    plt.tight_layout()
    plt.show()

    # Flag pairs that are suspiciously close to +-1 (possible redundant features or leakage)
    high_corr_pairs = [
        (numeric_cols[i], numeric_cols[j], corr.iloc[i, j])
        for i in range(len(numeric_cols))
        for j in range(i + 1, len(numeric_cols))
        if abs(corr.iloc[i, j]) > 0.9
    ]
    if high_corr_pairs:
        print("Very high correlations (|r| > 0.9) -- check these aren't duplicates or leaked features:")
        for a, b, r in high_corr_pairs:
            print(f"  {a} <-> {b}: r = {r:.3f}")
    else:
        print("No pairs with |correlation| > 0.9.")
else:
    print("Need at least 2 numeric columns for a correlation heatmap.")

In [ ]:
# Target column distribution -- essential for spotting class imbalance (classification)
# or a skewed target (regression) before you ever train a model
if TARGET_COLUMN and TARGET_COLUMN in df.columns:
    fig, ax = plt.subplots()
    if PROBLEM_TYPE == "classification" or df[TARGET_COLUMN].dtype == "object":
        df[TARGET_COLUMN].value_counts().plot(kind="bar", ax=ax)
        ax.set_title(f"Class frequencies for target: {TARGET_COLUMN}")
        ax.set_ylabel("count")
    else:
        df[TARGET_COLUMN].hist(bins=30, ax=ax)
        ax.set_title(f"Distribution of target: {TARGET_COLUMN}")
    plt.tight_layout()
    plt.show()
else:
    print(f"TARGET_COLUMN {TARGET_COLUMN!r} not found in the dataset (or not set) -- skipping target plot.")

## 4. Six Problems Hiding in Real Datasets

Every one of these is invisible in a model's accuracy score until it's too late — which is why inspection comes before modelling.

| Problem | What to do about it |
|---|---|
| **Missing values** | Drop the rows, drop the column, or impute with the median. Depends on how much is missing and why. |
| **Duplicate records** | Inflate counts and leak between train/test splits. Check before splitting, not after. |
| **Outliers** | A genuine extreme value or a data-entry error? Investigate before deleting anything. |
| **Wrong data types** | Numbers as text, dates as strings, categories as integers. Fix with `astype` / `to_datetime`. |
| **Class imbalance** | If 99% of rows are one class, a model predicting only that class scores 99% and is useless. |
| **Data leakage** | A feature that secretly contains the answer. Perfect scores in training, failure in production. |

The next cells run an automatic check for each one and print what it found, in the same order as the table.

In [ ]:
### Problem 1: Missing values
print("=== Missing values ===")
cols_with_missing = missing[missing > 0].sort_values(ascending=False)
if cols_with_missing.empty:
    print("None found. Nothing to clean here.")
else:
    for col, count in cols_with_missing.items():
        pct = count / len(df) * 100
        if pct > 50:
            verdict = "consider DROPPING this column -- more than half is missing"
        elif pct > 5:
            verdict = "impute (e.g. median for numeric, mode for categorical) or investigate why it's missing"
        else:
            verdict = "small enough to safely impute or drop the few affected rows"
        print(f"  {col}: {count} missing ({pct:.1f}%) -> {verdict}")

In [ ]:
### Problem 2: Duplicate records
print("=== Duplicate records ===")
if n_duplicates == 0:
    print("None found.")
else:
    dup_pct = n_duplicates / len(df) * 100
    print(f"{n_duplicates} duplicate rows ({dup_pct:.1f}% of the dataset).")
    print("-> Drop with df.drop_duplicates() BEFORE splitting into train/test,")
    print("   otherwise the same row can end up in both splits and leak information.")

In [ ]:
### Problem 3: Outliers (using the IQR rule -- same idea as the box plot whiskers above)
print("=== Outliers (IQR method) ===")
outlier_summary = {}
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = n_outliers

if not outlier_summary or all(v == 0 for v in outlier_summary.values()):
    print("No IQR outliers detected in any numeric column.")
else:
    for col, n in sorted(outlier_summary.items(), key=lambda kv: -kv[1]):
        if n > 0:
            pct = n / len(df) * 100
            print(f"  {col}: {n} potential outliers ({pct:.1f}%)")
    print("\n-> Look at these rows individually: a genuine extreme value (keep it) or a")
    print("   data-entry error (fix or drop it)? Don't delete automatically.")
    print("\nNote: the IQR rule assumes a roughly continuous column. Binary flags (0/1)")
    print("or columns with very few unique values will often show up here as a false")
    print("positive -- ignore those and focus on genuinely continuous columns.")

In [ ]:
### Problem 4: Wrong data types
print("=== Wrong data types ===")
suspects = []
for col in df.select_dtypes(include="object").columns:
    sample = df[col].dropna().astype(str).head(200)
    if sample.empty:
        continue
    # looks numeric but stored as text?
    looks_numeric = sample.str.replace(",", "", regex=False).str.replace(
        r"^[\$\-]?\d+\.?\d*$", "NUMERIC", regex=True
    ).eq("NUMERIC").mean()
    # looks like a date but stored as text?
    looks_date = pd.to_datetime(sample, errors="coerce", format="mixed").notna().mean()
    if looks_numeric > 0.9:
        suspects.append((col, "looks numeric but is stored as text -- check for $, commas, or stray symbols"))
    elif looks_date > 0.9:
        suspects.append((col, "looks like a date but is stored as text -- use pd.to_datetime()"))

if not suspects:
    print("No obvious type mismatches found in text columns.")
else:
    for col, note in suspects:
        print(f"  {col}: {note}")

In [ ]:
### Problem 5: Class imbalance (classification problems only)
print("=== Class imbalance ===")
if PROBLEM_TYPE == "classification" and TARGET_COLUMN in df.columns:
    class_counts = df[TARGET_COLUMN].value_counts(normalize=True)
    print(class_counts.round(3))
    majority_pct = class_counts.iloc[0] * 100
    if majority_pct > 90:
        print(f"\nSevere imbalance: the majority class is {majority_pct:.1f}% of the data.")
        print("-> A model that always predicts this class would score that % on accuracy")
        print("   and be useless. Use precision/recall/F1 instead, and consider")
        print("   resampling (SMOTE, class_weight='balanced') during modelling.")
    elif majority_pct > 70:
        print(f"\nModerate imbalance: majority class is {majority_pct:.1f}%. Keep an eye on")
        print("   recall for the minority class, don't rely on accuracy alone.")
    else:
        print("\nReasonably balanced. Accuracy is a safer metric here, though F1 is still fine.")
elif PROBLEM_TYPE != "classification":
    print(f"Skipped -- PROBLEM_TYPE is {PROBLEM_TYPE!r}, class imbalance applies to classification only.")
else:
    print(f"TARGET_COLUMN {TARGET_COLUMN!r} not found -- set it in the config cell to check this.")

In [ ]:
### Problem 6: Data leakage
# This one can't be fully automated -- it needs your domain knowledge of what each
# column means. This cell flags the two most common *symptoms*:
#   (a) a feature that is suspiciously, almost perfectly correlated with the target
#   (b) a column name that suggests it's only known *after* the outcome happens
print("=== Data leakage (partial, automatic check only) ===")

leakage_name_hints = ["result", "outcome", "label", "final", "post_", "after_", "_actual"]
suspect_names = [
    col for col in df.columns
    if col != TARGET_COLUMN and any(hint in col.lower() for hint in leakage_name_hints)
]
if suspect_names:
    print("Columns whose NAME suggests they might be known only after the outcome:")
    for col in suspect_names:
        print(f"  - {col}")
else:
    print("No suspicious column names found by this simple keyword check.")

if TARGET_COLUMN in numeric_cols and len(numeric_cols) > 1:
    target_corr = df[numeric_cols].corr(numeric_only=True)[TARGET_COLUMN].drop(TARGET_COLUMN)
    near_perfect = target_corr[target_corr.abs() > 0.95]
    if not near_perfect.empty:
        print("\nFeatures suspiciously close to perfectly correlated with the target:")
        print(near_perfect.round(3))
        print("-> If a feature can only be known AFTER the target is decided in real life,")
        print("   it's leakage -- drop it, however good it makes your score look.")
    else:
        print("\nNo numeric feature is near-perfectly correlated with the target.")

print("\nReminder: the strongest leakage check is reading your column list and asking,")
print("for each one, 'would I actually have this value at prediction time?'")

## 5. Verdict: Is This Dataset Ready?

This cell turns everything above into a single scored checklist. It is a starting point, not a substitute for judgment — a dataset can pass every automatic check here and still be a bad choice for reasons only you know (e.g. it doesn't actually answer an interesting question, or you can't get more of it).

**Checks performed:**
- Enough rows to split into train/test and still have a meaningful sample.
- No column is missing more than 50% of its values.
- No exact duplicate rows remain.
- A target column exists (if you're doing supervised learning) and isn't dangerously imbalanced.
- No feature is suspiciously, near-perfectly correlated with the target (leakage smell).

Each check is worth points; the total decides the verdict.

In [ ]:
checks = []  # list of (check_name, passed: bool, detail: str)

# Check 1: enough rows
MIN_ROWS = 100  # a rough teaching-project floor; real projects often want far more
n_rows = len(df)
checks.append((
    "Enough rows",
    n_rows >= MIN_ROWS,
    f"{n_rows} rows (need >= {MIN_ROWS})",
))

# Check 2: no column missing more than half its values
worst_missing_pct = (missing / len(df) * 100).max() if len(df) else 100
checks.append((
    "No column >50% missing",
    worst_missing_pct <= 50,
    f"worst column is {worst_missing_pct:.1f}% missing",
))

# Check 3: no duplicate rows remaining
checks.append((
    "No duplicate rows",
    n_duplicates == 0,
    f"{n_duplicates} duplicate rows found" if n_duplicates else "none found",
))

# Check 4: target column present and not dangerously imbalanced (classification only)
if TARGET_COLUMN and TARGET_COLUMN in df.columns:
    if PROBLEM_TYPE == "classification":
        majority_pct = df[TARGET_COLUMN].value_counts(normalize=True).iloc[0] * 100
        checks.append((
            "Target present & not severely imbalanced",
            majority_pct <= 90,
            f"majority class is {majority_pct:.1f}% of rows (want <= 90%)",
        ))
    else:
        checks.append(("Target column present", True, f"{TARGET_COLUMN!r} found"))
elif PROBLEM_TYPE == "unsupervised":
    checks.append(("Target column present", True, "not required for unsupervised learning"))
else:
    checks.append((
        "Target column present",
        False,
        f"{TARGET_COLUMN!r} not found in the dataset -- fix TARGET_COLUMN in the config cell",
    ))

# Check 5: no near-perfect correlation with target (leakage smell)
if TARGET_COLUMN in numeric_cols and len(numeric_cols) > 1:
    target_corr = df[numeric_cols].corr(numeric_only=True)[TARGET_COLUMN].drop(TARGET_COLUMN)
    max_corr = target_corr.abs().max()
    checks.append((
        "No near-perfect target correlation",
        max_corr <= 0.95,
        f"highest |correlation| with target is {max_corr:.3f} (want <= 0.95)",
    ))
else:
    checks.append(("No near-perfect target correlation", True, "not checkable (no numeric target/features)"))

# --- Print the scorecard ---
passed = sum(1 for _, ok, _ in checks if ok)
total = len(checks)

print(f"{'CHECK':38s} {'RESULT':8s} DETAIL")
print("-" * 90)
for name, ok, detail in checks:
    print(f"{name:38s} {'PASS' if ok else 'FAIL':8s} {detail}")

print("-" * 90)
print(f"Score: {passed}/{total}")

if passed == total:
    print("\nVERDICT: READY -- this dataset passes every automatic check. Move on to feature")
    print("engineering and modelling (see notebook 01, Section 5).")
elif passed >= total - 1:
    print("\nVERDICT: ALMOST READY -- fix the single FAIL above, re-run this cell, then proceed.")
else:
    print("\nVERDICT: NOT READY -- multiple issues found. Address the FAILs above before modelling,")
    print("or consider whether a different dataset would save you time.")

## Summary & Next Steps

You just walked through the exact sequence taught in Part II:

1. **Load** it correctly (Section 1) — right reader, right encoding, right separator.
2. **Inspect** it in the first ten minutes (Section 2) — shape, head, info, describe, missing, duplicates, value counts, dtypes.
3. **Visualize** it (Section 3) — histograms, box plots, correlation heatmap, target distribution.
4. **Hunt** for the six classic problems (Section 4) — missing values, duplicates, outliers, wrong types, class imbalance, leakage.
5. **Get a verdict** (Section 5) — a scored go/no-go before you commit time to modelling.

**If your verdict was READY or ALMOST READY:** copy your cleaned `df` into a new notebook and continue with the ML workflow — feature engineering, `train_test_split`, `fit`, evaluate (see `01_introduction_to_machine_learning.ipynb`, Section 5, for a full worked example on synthetic data).

**If your verdict was NOT READY:** don't panic — this is normal, and it's *why* this step exists. Either fix what's flagged (drop bad columns, re-encode dates, handle missing values) and re-run this notebook, or decide the dataset itself isn't a good fit and look for another one. Either way, you found this out in ten minutes instead of after a week of modelling.